## Overture Maps Foundation
According to their webpage, Overture creates reliable, easy-to-use, and interoperable open map data.
They build their maps collaboratively incorporating map data from multiple sources including Overture Members, civic organizations, and open data sources.

Overture deals with the problem that multiple datasets reference the same real-world entities using their own conventions and vocabulary, making them difficult to merge and combine. They simplify interoperability by providing a system that links entities from different data sets to the same real-world entities.

Overture is currently focused on building the following core data layers:
- administrative boundaries
- base: water, land, land use, infrastructure, land cover
- buildings
- places
- transportation

More information about their data can be found in their [documentation](https://docs.overturemaps.org/).

### Buildings layer

The Overture Maps buildings theme describes human-made structures with roofs or interior spaces that are permanently or semi-permanently in one place (source: OSM building definition). The theme includes two feature types:

- **building**: The most basic form of a building feature. The geometry is expected to be the most outer footprint -- or roofprint if traced from satellite/aerial imagery -- of a building.
- **building_part**: A single part of a building. Building parts may have the same properties as buildings. A building part is associated with a parent building via a building_id.

The documentation offers a detailed explanation about the data schema for each of the layers. For the building layer, it can be read [here](https://docs.overturemaps.org/schema/reference/buildings/building/).

The OMF’s Buildings layer includes over 780M unique building footprints worldwide. This layer has been developed by combining various open data projects including OpenStreetMap, Microsoft AI-Generated building footprints, and Esri. [Source](https://docs.overturemaps.org/guides/buildings/#14/32.58453/-117.05154/0/60).

The following example downloads building data from Overture using their Python package, which is a Beta version.

In [2]:
# !pip install geopandas overturemaps lonboard folium

import pandas as pd
import geopandas as gpd
import overturemaps 
from shapely import wkb
from lonboard import Map, PolygonLayer
from lonboard.colormap import apply_categorical_cmap
import folium
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 500) # Allows to inspect all the columns present in the dataframe

In [3]:
# Specify bounding box
east, south, west, north = (36.62768265193316, 37.561544985221865, 37.76386851796805, 38.460405921314695)
bbox = east, south, west, north # Use the 50km rectangle

In [4]:
# Download builidngs from overture
table = overturemaps.record_batch_reader("building", bbox).read_all()
table = table.combine_chunks()
# Convert to dataframe
df = table.to_pandas()

In [5]:
# Dataframe to geodataframe, set crs
gdf_overture = gpd.GeoDataFrame(
    df, 
    geometry=df['geometry'].apply(wkb.loads), 
    crs="EPSG:4326"
)

In [7]:
gdf_overture.head()

,id,names,sources,level,height,min_height,is_underground,num_floors,num_floors_underground,min_floor,subtype,class,facade_color,facade_material,roof_material,roof_shape,roof_direction,roof_orientation,roof_color,roof_height,geometry,has_parts,version,bbox
0,27f9713e-23b8-4c77-a736-bcd6059f4afd,None,"[{'property': '', 'dataset': 'Microsoft ML Bui...",NaN,NaN,NaN,False,NaN,NaN,NaN,None,None,None,None,None,None,NaN,None,None,NaN,"POLYGON ((36.66467 37.66139, 36.66467 37.66142...",False,3,"{'xmin': 36.66429901123047, 'xmax': 36.6646728..."
1,2ab73f75-8c5c-4289-9003-fa46d31d77bb,None,"[{'property': '', 'dataset': 'Microsoft ML Bui...",NaN,NaN,NaN,False,NaN,NaN,NaN,None,None,None,None,None,None,NaN,None,None,NaN,"POLYGON ((36.70309 38.16212, 36.70313 38.16224...",False,3,"{'xmin': 36.702945709228516, 'xmax': 36.703128..."
2,2d20c49b-00a6-4381-89ae-32757aac3c0b,None,"[{'property': '', 'dataset': 'Microsoft ML Bui...",NaN,NaN,NaN,False,NaN,NaN,NaN,None,None,None,None,None,None,NaN,None,None,NaN,"POLYGON ((36.69004 37.81711, 36.69004 37.81723...",False,3,"{'xmin': 36.68991470336914, 'xmax': 36.6900482..."
3,2f98f5a9-597d-499e-8382-1111e784c6c2,None,"[{'property': '', 'dataset': 'Microsoft ML Bui...",NaN,NaN,NaN,False,NaN,NaN,NaN,None,None,None,None,None,None,NaN,None,None,NaN,"POLYGON ((36.68529 38.16934, 36.68513 38.16933...",False,3,"{'xmin': 36.68513107299805, 'xmax': 36.6852989..."
4,3b2b4811-6f4e-47e4-b091-a8503c4450a0,None,"[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,NaN,NaN,False,NaN,NaN,NaN,None,None,None,None,None,None,NaN,None,None,NaN,"POLYGON ((36.75592 38.13026, 36.75601 38.13023...",False,1,"{'xmin': 36.75592041015625, 'xmax': 36.7560424..."


Overture uses different sources to create their maps data. Below, it can be seen that for this region, footprints come from OpenStreet Map and from Microsoft ML Buildings. 

Below, there is a map that colors the footprint based on its source.

In [8]:
# extract the data point primary source to compare with OSM
gdf_overture['dataset'] = gdf_overture.sources.apply(lambda x: x[0]['dataset'])

In [9]:
gdf_overture.dataset.value_counts()

OpenStreetMap             161240
Microsoft ML Buildings     56718
Name: dataset, dtype: int64

In [10]:
gdf_overture[['geometry', 'dataset']].iloc[0:1000].explore(column = 'dataset', tiles = 'CartoDB Positron')

### Places
The Overture Maps places theme contains datasets with point representations of real-world facilities, services, businesses, or amenities.

OMF’s Places data layer includes over 59 million POI records that have not previously been released as open data. This dataset, derived from data contributed to OMF by founding members **Meta** and **Microsoft**, provides a significant baseline of worldwide places data. The Overture community will combine the best data from all available resources, including open government data, crowdsourced local mapping data, AI/ML techniques and more to improve, update and extend the data on an ongoing basis.  The Places dataset is licensed under CDLA Permissive v2.0, a permissive open data license, and can be freely used by any map builders or location service providers.

The following example shows how to download the place layer for the area of interest. 

In [11]:
# Download builidngs from overture
table_places = overturemaps.record_batch_reader("place", bbox).read_all()
table_places = table_places.combine_chunks()
# Convert to dataframe
df = table_places.to_pandas()

In [12]:
# Dataframe to geodataframe, set crs
gdf_overture_plc = gpd.GeoDataFrame(
    df, 
    geometry=df['geometry'].apply(wkb.loads), 
    crs="EPSG:4326"
)

In [21]:
# Extract the data point primary source to compare with OSM
gdf_overture_plc['dataset'] = gdf_overture_plc.sources.apply(lambda x: x[0]['dataset'])

# Extract the category
gdf_overture_plc.loc[gdf_overture_plc['categories'].notnull(), 'category'] = gdf_overture_plc.loc[gdf_overture_plc['categories'].notnull(), 'categories'].apply(lambda x: x['primary'])

# Extract the primary name
gdf_overture_plc['primary_name'] = gdf_overture_plc.names.apply(lambda x: x['primary'])

In [22]:
gdf_overture_plc.head()

,id,geometry,categories,confidence,websites,emails,socials,phones,brand,addresses,names,sources,operating_status,basic_category,taxonomy,version,bbox,dataset,category,primary_name
0,0bbd2945-a04b-41d7-a44b-3fe22bb40ec4,POINT (36.96241 37.56337),"{'primary': 'coworking_space', 'alternate': None}",0.770000,[http://www.inovamuhendislik.com.tr],None,None,[(0344) 236 39 59],None,[{'freeform': 'Yeni San. Sit. Hızarcılar Cad.'...,"{'primary': 'İnova Mühendislik', 'common': Non...","[{'property': '', 'dataset': 'Foursquare', 'li...",open,b2b_service,"{'primary': 'coworking_space', 'hierarchy': ['...",3,"{'xmin': 36.96240997314453, 'xmax': 36.9624137...",Foursquare,coworking_space,İnova Mühendislik
1,1c80f57c-2adb-42aa-b661-e8da9d8b9966,POINT (36.91455 37.57042),"{'primary': 'banks', 'alternate': None}",0.770000,[https://www.garantibbva.com.tr],None,"[https://www.twitter.com/garantibbva, https://...",None,None,"[{'freeform': None, 'locality': None, 'postcod...","{'primary': 'Garanti BBVA', 'common': None, 'r...","[{'property': '', 'dataset': 'Foursquare', 'li...",open,bank,"{'primary': 'bank', 'hierarchy': ['services_an...",3,"{'xmin': 36.914546966552734, 'xmax': 36.914554...",Foursquare,banks,Garanti BBVA
2,a821ddab-6101-4a73-8086-0bd8bd4b8b06,POINT (36.90272 37.58492),"{'primary': 'boutique', 'alternate': None}",0.938874,[http://www.zehracicek.com/],[zehracicek4646@gmail.com],[https://www.facebook.com/203874742820091],[+905303115033],"{'wikidata': None, 'names': {'primary': None, ...","[{'freeform': 'Adnan Menderes Bulvarı 51D', 'l...","{'primary': 'Zehra Çiçek Exclusive', 'common':...","[{'property': '', 'dataset': 'meta', 'license'...",open,fashion_and_apparel_store,"{'primary': 'fashion_boutique', 'hierarchy': [...",8,"{'xmin': 36.90271759033203, 'xmax': 36.9027214...",meta,boutique,Zehra Çiçek Exclusive
3,effc72ac-0c0c-4fe2-b08a-4e958ab7bf04,POINT (36.87735 37.60097),"{'primary': 'campground', 'alternate': None}",0.936728,None,None,[https://www.facebook.com/390376148372123],None,"{'wikidata': None, 'names': {'primary': None, ...","[{'freeform': '', 'locality': 'Onikişubat', 'p...","{'primary': 'Ali Kayası Seyir', 'common': None...","[{'property': '', 'dataset': 'meta', 'license'...",open,campground,"{'primary': 'campground', 'hierarchy': ['lodgi...",5,"{'xmin': 36.877342224121094, 'xmax': 36.877349...",meta,campground,Ali Kayası Seyir
4,e48070f5-95c2-476b-8c98-f96722cdb2fd,POINT (36.85519 37.57627),"{'primary': 'arts_and_crafts', 'alternate': None}",0.936728,None,None,[https://www.facebook.com/262945954418490],None,"{'wikidata': None, 'names': {'primary': None, ...","[{'freeform': '', 'locality': 'Onikişubat', 'p...","{'primary': 'New Estetik Ve Güzellik Salonu', ...","[{'property': '', 'dataset': 'meta', 'license'...",open,arts_crafts_and_hobby_store,"{'primary': 'arts_and_crafts_store', 'hierarch...",6,"{'xmin': 36.85518264770508, 'xmax': 36.8551864...",meta,arts_and_crafts,New Estetik Ve Güzellik Salonu


The code below explores:
- Where the data comes from. In this case, it is from Meta and Microsoft. 
- The categories, a list with all OMF categories can be found [here](https://github.com/OvertureMaps/schema/blob/main/docs/schema/concepts/by-theme/places/overture_categories.csv). This example is interested in extracting health care-related facilities.
- The confidence of the existence of the place. Confidence is a number between 0 and 1. 0 means that OMF is sure that the place doesn't exist (anymore). 1 means that OMF is sure that the place exists. If there's no value for confidence, it means that OMF doesn't have any confidence information.

In [23]:
gdf_overture_plc.dataset.value_counts()

meta            9631
Foursquare       462
AllThePlaces     265
Microsoft        132
Name: dataset, dtype: int64

In [25]:
for cat in gdf_overture_plc.categories.unique():
    if cat == cat:
        if ('hospital' in cat) or ('health' in cat):
            print(cat)

TypeError: unhashable type: 'dict'

In [26]:
# Create a map to show health related facilities
m = gdf_overture_plc[gdf_overture_plc['category'].isin(['health_and_medical', 'hospital', 'counseling_and_mental_health'])][['category', 'dataset', 'geometry', 'primary_name', 'confidence', 'id']].explore(column = 'category', name = 'overture', tiles='CartoDB positron', cmap = 'viridis')
folium.LayerControl().add_to(m)
m

In [ ]:
# Create a map to show health related facilities that have a confidence greater than 0.5
m = gdf_overture_plc[(gdf_overture_plc['category'].isin(['health_and_medical', 'hospital', 'counseling_and_mental_health'])) &
                     (gdf_overture_plc['confidence']>0.5)][['category', 'dataset', 'geometry', 'primary_name', 'confidence', 'id']].explore(column = 'category', name = 'overture', tiles='CartoDB positron', cmap = 'viridis')
folium.LayerControl().add_to(m)
m

## Practice
1. Download the building footprints for a region of your interest, where the 4 introduced datasets are available, and compare the results.
2. Download hospitals and education institutions for Rio Grande do Sul State in Brasil.